# 06 · Failure cases and conclusions

Phases 7–8: look closely at where the baseline fails, and what that means for the capstone. The scores are in `05_evaluation.ipynb`; this notebook explains them.

In [1]:
import sys
sys.path.append("../src")

import re
import pandas as pd
from data import load_case_law, load_slice, clean_slice
from baseline import extract
from evaluate import gold_spans, predicted_spans, match

In [2]:
case_law = load_case_law()
df, _ = clean_slice(load_slice(case_law), case_law)

sample = pd.read_csv("../data/eval_sample.csv").merge(df[["chunk_id", "text"]], on="chunk_id")
labels = pd.read_csv("../data/labels.csv", keep_default_na=False)
gold = gold_spans(sample, labels)
pred = predicted_spans(sample)
gold_p, _ = match(gold, pred, "partial")

def context(item, phrase, width=70):
    """The phrase with some text on either side, from the chunk it appears in."""
    text = sample.loc[sample["item"] == item, "text"].iloc[0]
    i = text.lower().find(phrase.lower())
    return "…" + text[max(0, i - width): i + len(phrase) + width] + "…"

## Failure case 1: spaced ordinals hide every missed date

All 4 missed dates follow one convention: a space between the day number and its ordinal suffix.

In [3]:
gold_p[~gold_p["matched"] & (gold_p["type"] == "DATE")][["item", "text"]]

,item,text
5,L02,24 th February 2025
17,L07,14 th February 2020
28,L11,"20 th July, 2015"
35,L19,"23 RD DAY OF MARCH, 2026"


In [4]:
print(context("L02", "24 th February 2025"))
print()
print(context("L19", "23 RD DAY OF MARCH, 2026"))
print()
print("baseline output for the L19 phrase:", extract("THIS 23 RD DAY OF MARCH, 2026"))

…ity Wanjiku Kariuki 2 nd Defendant Ruling 1. The application is dated 24 th February 2025 and is brought under Order 40 Rule 1 and 2 and Order 51 Rule 1 of the…

…to the respondent herein. DATED, SIGNED AND DELIVERED AT ELDORET THIS 23 RD DAY OF MARCH, 2026. EMMANUEL M.WASHE JUDGE In The Presence Of : Court Assistant: Mr.Bria…

baseline output for the L19 phrase: []


**Why it fails.** The `day_month` pattern expects the suffix attached to the number (`24th`), and `DAY OF` between the day and the month isn't allowed for at all. It's not rare: the cleaning notebook counted spaced ordinals in 114 of the 381 chunks. Most are party labels (`1 st Defendant`), but the same habit shows up in dates. Allowing an optional space before `st/nd/rd/th` would recover three of the four misses without catching party labels, because the pattern still needs a month name straight after. The `23 RD DAY OF MARCH` form needs a pattern of its own.

**Why it matters.** L19 is a judgment's **delivery date**, one of the most important dates in any judgment, and it's written in the formal style Kenyan courts use to close a judgment. A lawyer who asked "when was this delivered?" would get nothing.

## Failure case 2: law-report citations outside the `[YYYY] CODE N` shape

7 of the 19 case references are missed completely, and all 7 are citations whose shape the `neutral` pattern doesn't expect.

In [5]:
gold_p[~gold_p["matched"] & (gold_p["type"] == "CASE_REF")][["item", "text"]]

,item,text
8,L03,(2007] KECA 115 [KLR)
10,L03,[2024) KEELC 1505 (KLR])
29,L12,[1931] 47 TLK 557
31,L12,[2003] 2 EA 519
33,L13,[1972] ALL ER 606
34,L17,[2004] 2 EA 163
36,L20,[1908] 24 T.L.R. 548


In [6]:
print(context("L12", "[2003] 2 EA 519", 90))

…er of the motor vehicle. In the case of Kenya Bus Services Ltd v Humphrey [2003] KLR 665; [2003] 2 EA 519, the Court of Appeal cited Kansa v Solanki [1969] EA 318 wherein it was held that: “ Wher…


**Why it fails.** The pattern requires `[year]`, then letters, then a number. Real citations break this in three ways:
- **Volume number first:** `[2003] 2 EA 519`, `[2004] 2 EA 163`. The East Africa Law Reports (EA) are cited like this, and they are common in Kenyan judgments.
- **Multi-word or dotted report names:** `[1972] ALL ER 606`, `[1908] 24 T.L.R. 548`, older English reports that Kenyan courts still cite.
- **Garbled source text:** `(2007] KECA 115 [KLR)` in L03. That chunk's brackets are scrambled throughout, probably from OCR of a scanned PDF.

Notice that in L12 the pattern matched `[2003] KLR 665` but missed `[2003] 2 EA 519` right beside it: the *same case*, cited in two reports.

**Why it matters.** Citations to decided cases are central to legal research: they are how a lawyer follows precedent. Missing the older and foreign reports removes exactly the authorities that are hardest to find by other means.

## Failure case 3: partial spans that look right but aren't

These were counted as correct by partial matching. Two of them change what the extracted entity means.

In [7]:
print(context("L10", "Kshs 4, 000"))
print("baseline output:", [e["text"] for e in extract(sample.loc[sample["item"] == "L10", "text"].iloc[0])])

…ct evidence to show the accused person as the offender. The purported Kshs 4, 000 recovered from the accused and which was not produced in court bore n…
baseline output: ['Kshs 4']


In [8]:
print(context("L02", "Land Case E018 of 2025", 40))
print("baseline output:", [e["text"] for e in extract("Land Case E018 of 2025")])

…Mula & 2 others v Mutuma & another (Land Case E018 of 2025) [2026] KEELC 1791 (KLR) (25 March 2026…
baseline output: ['Case E018 of 2025']


**Why it fails.**
- **`Kshs 4, 000` → `Kshs 4`.** The pattern allows only a comma directly followed by three digits, so the stray space in the source ends the match early. The result is a well-formed but **wrong** amount: KES 4 instead of KES 4,000.
- **`Land Case E018 of 2025` → `Case E018 of 2025`.** `Land` isn't in the keyword list, so the match starts at `Case`. The same happens with `High Court Civil Case` → `Civil Case`. The case number is right, but the court or division that identifies it is lost, and case numbers are only unique within a court and division.

**Why it matters.** A wrong amount is worse than a missing one, because nothing signals the error. This is also a lesson about the scoring: the partial MONEY F1 of 0.92 hides this, and the exact F1 of 0.77 is the more honest number for money. For the capstone, amounts should be scored on their **value**, not just on overlapping text.

## Other misses worth noting

- **`9.000` in L09** ("paid Ksh.8,500 and later to 9.000 and Kshs.10,000"): a shilling amount with no currency marker and a dot as the thousands separator. Only the context shows it's money. A regex can't read context.
- **Entities split at chunk boundaries:** none of the 37 gold entities in this sample were cut off, but 330 of the 381 cleaned chunks start mid-sentence, so it will happen in a larger sample.

## Limitations of the baseline

| Limitation | Evidence |
|---|---|
| Only recognises formats written into it | All 4 missed dates are one unhandled format; all 7 missed citations are unhandled citation shapes |
| No knowledge of context | Can't tell a delivery date from a date of birth, or tell `9.000` is money |
| Fragile to messy source text | OCR-garbled brackets (L03) and a stray space inside a number (L10) break matches |
| Returns strings, not values | `Kshs 4` looks like a valid amount; nothing checks it |
| Keyword lists are never complete | `Land Case`, `Matrimonial Cause`, `Originating Summons` all have missing keywords |
| Works per chunk | An entity split across two chunks can't be recovered |

## Limitations of the evaluation

- 20 chunks and 37 gold entities, so each entity moves recall by about 3 points.
- The 12/8 split under-represents chunks where the baseline found nothing (40% in the sample vs 60% in the slice), so recall is probably **overestimated**.
- No false positives appeared, but 25 predictions is too few to show that precision really is high.
- One labeller (AI-drafted, author-reviewed), no second labeller to check agreement.
- Mostly Magistrates' Courts, like the slice itself.

## What this means for the capstone

**The baseline to beat:** exact F1 **0.61** overall (DATE 0.78, MONEY 0.77, CASE_REF 0.45), with recall (0.51) as the main weakness. For a tool lawyers rely on, missed dates and citations are the costly error, so recall is the number the capstone system has to raise without giving up precision.

**What the failures point to:**
1. **Normalise the text before extracting.** Joining spaced ordinals (`24 th` → `24th`) and fixing spacing inside numbers is cheap. On this sample it would fix 3 of the 4 date misses and the `Kshs 4, 000` error; the fourth (`23 RD DAY OF MARCH`) also needs a pattern for the formal delivery line. It belongs in the cleaning step.
2. **Case references need more than a regex.** The citation shapes vary too much (volumes, report series, OCR damage) for hand-written patterns to keep up. This is where a learned model is worth trying, for example a token-classification NER model or an LLM with structured output, compared against this baseline on the same labels.
3. **Extract values, not just spans.** Amounts should be parsed to numbers and dates to calendar dates, and scored on the value. Otherwise a model can look right while being wrong, as `Kshs 4` shows.
4. **Work at the judgment level, not the chunk level.** Rejoining a judgment's chunks before extraction removes boundary splits and lets the system use context (e.g. "delivered on" before a date).
5. **Build a better test set before comparing models.** Around 100 chunks, sampled randomly rather than split by baseline output, with a second labeller on a subset to measure agreement. The current 20 chunks are enough to show where the baseline fails, not to rank competing systems.

**In one line:** regex gets about half the entities with high precision, and fails systematically, not randomly: on formatting conventions, citation variety and messy source text. That makes the failures predictable, and the capstone's first improvements can target them directly.